# ISIC MILK10k Benchmark — Training Notebook

**Task:** Multi-category skin lesion diagnosis (11 classes)  
**Metric:** Macro F1 (threshold = 0.5)  

---
**Truoc khi chay:**
1. `Runtime > Change runtime type > T4 GPU`
2. Chay lan luot tung cell tu tren xuong

## Buoc 1 — Kiem tra GPU

In [ ]:
import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Khong tim thay GPU — doi Runtime > T4!")

## Buoc 2 — Clone / update project & cai dat thu vien

In [ ]:
import os

REPO_URL = "https://github.com/trong5nhan6/deeplearning.git"
REPO_DIR = "/content/deeplearning"
PROJ_DIR = f"{REPO_DIR}/MILK10K_SOLUTION"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(PROJ_DIR)
print(f"Working dir: {os.getcwd()}")

In [ ]:
%%capture
!pip install timm>=0.9.12 albumentations>=1.3.1 iterative-stratification tqdm gdown

In [ ]:
import timm, albumentations, sklearn
print(f"timm          : {timm.__version__}")
print(f"albumentations: {albumentations.__version__}")
print(f"scikit-learn  : {sklearn.__version__}")
print("Libraries OK")

## Buoc 3 — Tai dataset

In [ ]:
import os
from pathlib import Path

DATASET_DIR = Path("datasets/MILK10k")
ZIP_PATH    = Path("/content/MILK10k.zip")
FILE_ID     = "1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX"

if DATASET_DIR.exists():
    print("Dataset da ton tai")
else:
    print("Dang tai...")
    !gdown {FILE_ID} -O {ZIP_PATH}
    os.makedirs("datasets", exist_ok=True)
    !unzip -q {ZIP_PATH} -d datasets/
    ZIP_PATH.unlink(missing_ok=True)
    print("Xong")

for p in sorted(Path("datasets").rglob("*.csv")):
    print(" ", p)

## Buoc 4 — Chuan bi data (chi chay 1 lan)

In [ ]:
from pathlib import Path
import os

if not Path("datasets/MILK10k/train/train_combined.csv").exists():
    !python src/prepare_data.py --base_dir datasets/MILK10k
else:
    print("Combined CSVs da co san")

if not Path("datasets/MILK10k/splits/train_fold0.csv").exists():
    for fold in range(5):
        os.system(
            f"python src/split_data.py "
            f"--csv datasets/MILK10k/train/train_combined.csv "
            f"--out_dir datasets/MILK10k/splits "
            f"--fold {fold} --n_folds 5 --seed 42"
        )
    print("Splits xong")
else:
    print("Splits da co san")

In [ ]:
import pandas as pd

train_df = pd.read_csv("datasets/MILK10k/splits/train_fold0.csv")
val_df   = pd.read_csv("datasets/MILK10k/splits/val_fold0.csv")
test_df  = pd.read_csv("datasets/MILK10k/test/test_combined.csv")

print(f"Train : {len(train_df):,}  Val : {len(val_df):,}  Test : {len(test_df):,}")

LABEL_COLS = ["AKIEC","BCC","BEN_OTH","BKL","DF","INF","MAL_OTH","MEL","NV","SCCKA","VASC"]
print("\nClass distribution (train fold0):")
for c in LABEL_COLS:
    n = int(train_df[c].sum())
    print(f"  {c:<10}: {n:>5} ({100*n/len(train_df):>5.1f}%)")

## Buoc 5 — Cau hinh training

Chon model: `swin_base` | `convnext_base` | `efficientnet_b3` | `maxvit_tiny` | `vit_base`

In [ ]:
import yaml

CONFIG = "configs/swin_base.yaml"

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

cfg.update({
    "epochs"                  : 30,
    "batch_size"              : 32,
    "num_workers"             : 2,
    "image_size"              : 224,
    "lr"                      : 1e-4,
    "loss_name"               : "bce",
    "use_pos_weight"          : True,
    "use_amp"                 : True,
    "early_stopping_patience" : 8,
})

print(f"Model  : {cfg['model_name']}")
print(f"Epochs : {cfg['epochs']}  Batch : {cfg['batch_size']}  LR : {cfg['lr']}  Loss : {cfg['loss_name']}")

## Buoc 6 — Train model

In [ ]:
import sys
sys.path.insert(0, ".")

from src.dataset import build_meta_processor
from models.model_factory import build_model
from src.train import train
from src.utils import set_seed
import pandas as pd

set_seed(cfg.get("seed", 42))

meta_dim = 0
if cfg.get("use_metadata", False):
    proc     = build_meta_processor(pd.read_csv(cfg["train_csv"]))
    meta_dim = proc.meta_dim

model    = build_model(cfg, meta_dim=meta_dim)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model  : {cfg['model_name']}")
print(f"Params : {n_params:,}")

In [ ]:
# Log format: [Epoch]  Loss tr/val  Acc tr/val  F1 tr/val  lr  time  [BEST]
best_f1 = train(cfg, model)
print(f"\nBest val macro F1 = {best_f1:.4f}")

## Buoc 7 — Xem training log

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

model_name = cfg["model_name"]
log_df     = pd.read_csv(f"outputs/logs/{model_name}_train_log.csv")
print(log_df[["epoch","train_loss","val_loss","train_acc","val_acc","train_f1","macro_f1"]].tail(10).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train", color="steelblue")
axes[0].plot(log_df["epoch"], log_df["val_loss"],   label="val",   color="tomato")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

# Accuracy
axes[1].plot(log_df["epoch"], log_df["train_acc"], label="train", color="steelblue")
axes[1].plot(log_df["epoch"], log_df["val_acc"],   label="val",   color="tomato")
axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("Epoch")

# Val Macro F1 per-class at best epoch
f1_cols     = [c for c in log_df.columns if c.startswith("f1_")]
best_row    = log_df.loc[log_df["macro_f1"].idxmax()]
class_names = [c.replace("f1_", "") for c in f1_cols]
class_f1s   = [best_row[c] for c in f1_cols]
axes[2].barh(class_names, class_f1s, color="steelblue")
axes[2].set_title(f"Per-class F1 @ best epoch ({int(best_row['epoch'])})")
axes[2].set_xlim(0, 1)

plt.tight_layout()
plt.savefig(f"outputs/logs/{model_name}_training_curves.png", dpi=150)
plt.show()
print(f"Best macro F1 : {log_df['macro_f1'].max():.4f}  @epoch {int(best_row['epoch'])}")

## Buoc 8 — Inference & tao submission

In [ ]:
from src.infer import infer

model_name   = cfg["model_name"]
checkpoint   = f"outputs/checkpoints/{model_name}/best.pth"
test_csv     = "datasets/MILK10k/test/test_combined.csv"
test_img_dir = "datasets/MILK10k/test/MILK10k_Test_Input"
out_path     = f"outputs/submissions/submission_{model_name}.csv"

infer(
    config_path=CONFIG, checkpoint=checkpoint,
    test_csv=test_csv, out_path=out_path,
    image_dir=test_img_dir, use_tta=False,
)
print(f"Saved: {out_path}")

In [ ]:
import pandas as pd

sub = pd.read_csv(out_path)
print(f"Rows    : {len(sub)}")
print(f"NaN     : {sub.isnull().sum().sum()}")
print(f"In[0,1] : {((sub.iloc[:,1:] >= 0) & (sub.iloc[:,1:] <= 1)).all().all()}")
display(sub.head())

## Buoc 9 — TTA inference (optional)

In [ ]:
out_tta = f"outputs/submissions/submission_{model_name}_tta.csv"
infer(
    config_path=CONFIG, checkpoint=checkpoint,
    test_csv=test_csv, out_path=out_tta,
    image_dir=test_img_dir, use_tta=True,
)
print(f"TTA saved: {out_tta}")

## Buoc 10 — Train 5 folds / Cross-Validation (optional)

In [ ]:
import yaml, sys
sys.path.insert(0, ".")
from src.dataset import build_meta_processor
from models.model_factory import build_model
from src.train import train
from src.infer import infer
from src.utils import set_seed
import pandas as pd

with open(CONFIG) as f:
    base_cfg = yaml.safe_load(f)
base_cfg.update(cfg)

fold_f1s  = []
fold_subs = []

for fold in range(5):
    print(f"\n{'='*40}  FOLD {fold}/4  {'='*40}")
    cfg_fold = base_cfg.copy()
    cfg_fold["train_csv"]      = f"datasets/MILK10k/splits/train_fold{fold}.csv"
    cfg_fold["val_csv"]        = f"datasets/MILK10k/splits/val_fold{fold}.csv"
    cfg_fold["checkpoint_dir"] = f"outputs/checkpoints/fold{fold}"
    set_seed(42 + fold)

    meta_dim = 0
    if cfg_fold.get("use_metadata", False):
        proc     = build_meta_processor(pd.read_csv(cfg_fold["train_csv"]))
        meta_dim = proc.meta_dim

    model   = build_model(cfg_fold, meta_dim=meta_dim)
    best_f1 = train(cfg_fold, model)
    fold_f1s.append(best_f1)

    ckpt     = f"outputs/checkpoints/fold{fold}/{cfg_fold['model_name']}/best.pth"
    sub_path = f"outputs/submissions/submission_fold{fold}.csv"
    infer(config_path=CONFIG, checkpoint=ckpt, test_csv=test_csv,
          out_path=sub_path, image_dir=test_img_dir, use_tta=True)
    fold_subs.append(sub_path)
    print(f"Fold {fold} best F1: {best_f1:.4f}")

print(f"\nCV  : {[f'{f:.4f}' for f in fold_f1s]}")
print(f"Mean: {sum(fold_f1s)/len(fold_f1s):.4f}")

In [ ]:
from src.submission import ensemble_submissions

ensemble_submissions(
    paths=fold_subs, weights=None,
    out_path="outputs/submissions/submission_5fold_ensemble.csv",
)
print("5-fold ensemble saved")

## Buoc 11 — Ensemble nhieu model (optional)

In [ ]:
from src.submission import ensemble_submissions
from pathlib import Path

submissions_to_ensemble = [
    "outputs/submissions/submission_swin_base_patch4_window7_224_tta.csv",
    "outputs/submissions/submission_convnext_base_tta.csv",
    "outputs/submissions/submission_efficientnet_b3_tta.csv",
]

existing = [p for p in submissions_to_ensemble if Path(p).exists()]
print(f"Found {len(existing)}/{len(submissions_to_ensemble)} submissions")

if len(existing) >= 2:
    ensemble_submissions(
        paths=existing, weights=None,
        out_path="outputs/submissions/submission_multi_model_ensemble.csv",
    )
    print("Multi-model ensemble saved")

## Buoc 12 — Download submission

In [ ]:
from google.colab import files
from pathlib import Path
import os

subs = sorted(Path("outputs/submissions").glob("*.csv"))
print("Submissions:")
for s in subs:
    print(f"  {s.name}  ({s.stat().st_size/1024:.1f} KB)")

best_sub = f"outputs/submissions/submission_{cfg['model_name']}_tta.csv"
target   = best_sub if os.path.exists(best_sub) else (str(subs[-1]) if subs else None)
if target:
    files.download(target)
    print(f"Downloading: {target}")
else:
    print("Chua co submission nao")